In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

IMG_ROOT = '../data/Images'
IMG_SIZE = (128, 128)  # resize target for all models

## Load image paths and labels

In [ ]:
# Map folder names to canonical label strings
label_map = {
    'Negative': 'negative',
    'Neutral':  'neutral',
    'positive': 'positive',  # note: lowercase folder name
}

records = []
for folder, label in label_map.items():
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(folder_path, fname), 'label': label})

df = pd.DataFrame(records)
print(f'Total images: {len(df)}')
df.head()

## Class distribution

In [ ]:
print('Class distribution:')
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True).round(3))

## Preprocessing pipeline

Steps applied to every image before it enters any model:
1. Open with PIL and convert to RGB (handles grayscale, RGBA, palette-mode images uniformly)
2. Resize to `IMG_SIZE` (64×64) using Lanczos resampling
3. Convert to float32 NumPy array
4. Normalize pixel values to [0, 1]

In [ ]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype='float32') / 255.0
    return arr

# Sanity check
sample_path = df['path'].iloc[0]
sample_arr = load_image(sample_path)
print(f'Output shape: {sample_arr.shape}')   # should be (64, 64, 3)
print(f'Min: {sample_arr.min():.3f}  Max: {sample_arr.max():.3f}')

## Sample images by class

In [ ]:
classes = ['negative', 'neutral', 'positive']
n_samples = 4

fig, axes = plt.subplots(len(classes), n_samples, figsize=(12, 7))
for row, cls in enumerate(classes):
    subset = df[df['label'] == cls].sample(n_samples, random_state=42)
    for col, (_, record) in enumerate(subset.iterrows()):
        axes[row, col].imshow(load_image(record['path']))
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls.capitalize(), fontsize=11, rotation=90, labelpad=40, va='center')
plt.suptitle('Sample Images by Sentiment Class (after preprocessing)', fontsize=13)
plt.tight_layout()
plt.show()

## Train/test split

Identical parameters to the text pipeline so both pipelines are evaluated on a consistent 80/20 split.

In [ ]:
paths_train, paths_test, y_train, y_test = train_test_split(
    df['path'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f'Train: {len(paths_train)}  Test: {len(paths_test)}')
print('\nTrain class distribution:')
print(y_train.value_counts(normalize=True).round(3))
print('\nTest class distribution:')
print(y_test.value_counts(normalize=True).round(3))